
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
select * from customers

In [0]:
describe extended customers

Spark SQL has built-in functionality to directly interact with JSON data stored as strings
We can simply use the colon(:) syntax to traverse nested data structures

In [0]:
select customer_id, profile:first_name,profile:address:country
from customers

In [0]:
select profile from customers limit 1

##from_json & schema_of_json Function

In [0]:
create or replace temporary view parsed_customers as
select customer_id, from_json(profile, schema_of_json('{"first_name":"Susana","last_name":"Gonnely","gender":"Female","address":{"street":"760 Express Court","city":"Obrenovac","country":"Serbia"}}')) as profile_struct
from customers;
select * from parsed_customers;

In [0]:
describe parsed_customers

In [0]:
select customer_id, profile_struct.first_name, profile_struct.address.country
from parsed_customers;

In [0]:
create or replace temporary view customers_final as
select customer_id, profile_struct.address.*,
profile_struct.first_name as first_name,
profile_struct.last_name as last_name,
profile_struct.gender as gender
from parsed_customers;
select * from customers_final

## Explode function

In [0]:
select order_id,customer_id,books from orders

In [0]:
select order_id,customer_id,explode(books) from orders

##collect_set Function

In [0]:
select customer_id,
    collect_set(order_id) as order_set,
    collect_set(books.book_id) as books_set
from orders
group by customer_id

In [0]:
select customer_id,
    collect_set(books.book_id) as before_flatten,
    array_distinct(flatten(collect_set(books.book_id))) as after_flatten
from orders
group by customer_id

## Inner Join

In [0]:
create or replace view orders_enriched as
select * 
from (
    select *,explode(books) as book
    from orders
) o
inner join books b on o.book.book_id=b.book_id;

select * from orders_enriched

## Set Operations

In [0]:
create or replace temporary view orders_updates
as select * from parquet.`/Volumes/demoworkspace/default/bookstore_dataset/orders-new/`;

select * from orders
union
select * from orders_updates

In [0]:
select * from orders
intersect
select * from orders_updates

In [0]:
select * from orders
minus
select * from orders_updates

##Pivot clause

In [0]:
create or replace table transactions as
select * from(
    select 
        customer_id,
        book.book_id as book_id,
        book.quantity as quantity
    from orders_enriched
)pivot(
        sum(quantity) for book_id in ('B01','B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','B12')
    );

select * from transactions